# 🧠 Build Your Own GPT: Machine Translation with Transformers
### *A Hands-On Assignment for Understanding Transformer Architecture*

---

## 🎯 Assignment Overview

You have learned about the Transformer architecture — attention mechanisms, positional encodings, multi-head attention, encoder-decoder stacks, and everything in between. Now it's time to **build something real**.

In this assignment, you will train a **Transformer-based sequence-to-sequence model** for **English → Urdu machine translation** using the [Helsinki-NLP/opus-100](https://huggingface.co/datasets/Helsinki-NLP/opus-100) dataset.

Think of this as building a mini version of Google Translate — from scratch (almost).

---

## 📚 Learning Objectives

By completing this assignment, you will:

1. Understand how to prepare parallel corpora for sequence-to-sequence learning
2. Build and configure a Transformer model using PyTorch
3. Train the model and monitor learning curves
4. Evaluate translation quality using BLEU score
5. Investigate how architectural hyperparameters (number of heads, layers, etc.) affect performance

---

## ⚠️ Instructions

- Cells marked with **`# TODO`** require you to write the actual implementation.
- Cells marked with **`# GIVEN`** are provided — read and understand them, but do not modify.
- At the end of each section, answer the **reflection questions** in a Markdown cell.
- For the **hyperparameter experiments** (Section 6), you must run at least **3 different configurations** and compare them.

---

> 💡 **Tip:** Run each cell in order. If you get an error, read the message carefully — it usually tells you exactly what's wrong.

---
## Section 0: Environment Setup

We install and import all the libraries we need. This assignment relies on:
- **`datasets`** — to load the Helsinki EN-UR dataset from Hugging Face
- **`tokenizers`** — to build a custom BPE tokenizer
- **`torch`** — for the Transformer model
- **`sacrebleu`** — for BLEU score evaluation
- **`matplotlib`** — for plotting learning curves

In [ ]:
# GIVEN — Run this cell first to install dependencies
!pip install datasets tokenizers sacrebleu torch torchtext matplotlib tqdm -q

In [ ]:
# GIVEN — Imports
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import math
import time
import random
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

from datasets import load_dataset
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

import sacrebleu

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

---
## Section 1: Load and Explore the Dataset

We use the **Helsinki-NLP/opus-100** dataset, which contains parallel sentence pairs across many language pairs. We load the **English–Urdu (`en-ur`)** split.

**What is a parallel corpus?** It's a collection of texts in two languages where each sentence in one language is paired with its translation in the other. This is the standard training data format for supervised machine translation.

Each example in this dataset looks like:
```python
{'translation': {'en': 'Hello, how are you?', 'ur': 'ہیلو، آپ کیسے ہیں؟'}}
```

In [ ]:
# GIVEN — Load the dataset
print("Loading Helsinki-NLP/opus-100 en-ur dataset...")
dataset = load_dataset("Helsinki-NLP/opus-100", "en-ur")
print(dataset)

train_data = dataset['train']
val_data   = dataset['validation']
test_data  = dataset['test']

print(f"\nTrain size : {len(train_data)}")
print(f"Val size   : {len(val_data)}")
print(f"Test size  : {len(test_data)}")

In [ ]:
# GIVEN — Inspect a few examples
print("Sample translation pairs:\n")
for i in range(5):
    pair = train_data[i]['translation']
    print(f"EN: {pair['en']}")
    print(f"UR: {pair['ur']}")
    print("-" * 60)

In [ ]:
# TODO — Dataset Analysis
# Complete the function below to compute and plot the sentence length distribution
# for both English and Urdu sentences in the training set.
#
# Hint: use pair['translation']['en'].split() and pair['translation']['ur'].split()
# to count tokens (word-level is fine here for analysis)

def compute_length_stats(data, n_samples=5000):
    """
    Compute token length statistics for EN and UR sentences.
    Returns two lists: en_lengths, ur_lengths
    """
    en_lengths = []
    ur_lengths = []
    
    # TODO: Iterate over n_samples examples from `data`
    # Append the word-count of each English sentence to en_lengths
    # Append the word-count of each Urdu sentence to ur_lengths
    
    pass  # Remove this when you write your code
    
    return en_lengths, ur_lengths

en_lens, ur_lens = compute_length_stats(train_data)

# TODO: Plot histograms of en_lens and ur_lens side by side
# Label axes, add a title, use plt.show()

# TODO: Print the mean, median, and 95th percentile for each language
# Use np.mean, np.median, np.percentile

### 📝 Reflection — Section 1

Answer the following questions in this cell:

1. What is the average sentence length in English vs. Urdu? Are they similar or different? Why might this matter for training?
2. What `MAX_SEQ_LEN` would you choose based on the 95th percentile, and what's the trade-off of choosing a longer vs. shorter value?

> *(Write your answers here)*

---
## Section 2: Tokenization

Before feeding text into a neural network, we need to convert words into numbers. We train a **Byte-Pair Encoding (BPE)** tokenizer — the same family of tokenizers used in GPT-2, RoBERTa, and many modern LLMs.

**What is BPE?** It starts with individual characters and iteratively merges the most frequent adjacent pair into a new token. This gives us a vocabulary of sub-word units — great for morphologically rich languages like Urdu.

We train a **shared vocabulary** across both English and Urdu. This means the tokenizer sees both languages and learns sub-word units that might appear in either.

In [ ]:
# GIVEN — Build a shared BPE tokenizer on the training data

VOCAB_SIZE  = 16000
MAX_SEQ_LEN = 64    # Truncate/pad all sequences to this length

# Special tokens
PAD_TOKEN = "[PAD]"   # padding to make batches the same length
UNK_TOKEN = "[UNK]"   # unknown sub-word
SOS_TOKEN = "[SOS]"   # start-of-sequence (decoder input)
EOS_TOKEN = "[EOS]"   # end-of-sequence

# Collect all sentences from training split
all_sentences = []
for example in train_data:
    all_sentences.append(example['translation']['en'])
    all_sentences.append(example['translation']['ur'])

# Train BPE tokenizer
tokenizer = Tokenizer(BPE(unk_token=UNK_TOKEN))
tokenizer.pre_tokenizer = Whitespace()

trainer = BpeTrainer(
    vocab_size=VOCAB_SIZE,
    special_tokens=[PAD_TOKEN, UNK_TOKEN, SOS_TOKEN, EOS_TOKEN]
)
tokenizer.train_from_iterator(all_sentences, trainer=trainer)

PAD_IDX = tokenizer.token_to_id(PAD_TOKEN)
SOS_IDX = tokenizer.token_to_id(SOS_TOKEN)
EOS_IDX = tokenizer.token_to_id(EOS_TOKEN)

print(f"Vocabulary size : {tokenizer.get_vocab_size()}")
print(f"PAD index       : {PAD_IDX}")
print(f"SOS index       : {SOS_IDX}")
print(f"EOS index       : {EOS_IDX}")

In [ ]:
# TODO — Tokenizer Exploration
# Pick any English sentence from the dataset.
# 1. Tokenize it using: tokenizer.encode(sentence).tokens
# 2. Get the token IDs using: tokenizer.encode(sentence).ids
# 3. Print both and observe how BPE splits the words.
# 4. Do the same for its Urdu translation.
# What do you notice about how BPE handles Urdu script vs English?

sample_en = train_data[0]['translation']['en']
sample_ur = train_data[0]['translation']['ur']

# TODO: Encode and print tokens and IDs for both sentences


---
## Section 3: Dataset and DataLoader

We wrap the raw data in a PyTorch `Dataset` class that:
1. Tokenizes each sentence pair
2. Adds `[SOS]` and `[EOS]` tokens
3. Pads / truncates to `MAX_SEQ_LEN`

This is the standard pipeline for feeding variable-length text into a batched training loop.

In [ ]:
# GIVEN — Dataset class

def encode_sentence(sentence, max_len):
    """Tokenize, add SOS/EOS, and pad/truncate to max_len."""
    ids = tokenizer.encode(sentence).ids
    ids = [SOS_IDX] + ids[:max_len - 2] + [EOS_IDX]
    # Pad to max_len
    ids += [PAD_IDX] * (max_len - len(ids))
    return ids


class TranslationDataset(Dataset):
    def __init__(self, hf_dataset, max_len=MAX_SEQ_LEN):
        self.data    = hf_dataset
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        pair = self.data[idx]['translation']
        src  = torch.tensor(encode_sentence(pair['en'], self.max_len), dtype=torch.long)
        tgt  = torch.tensor(encode_sentence(pair['ur'], self.max_len), dtype=torch.long)
        return src, tgt


# Create dataset and dataloaders
BATCH_SIZE = 64

train_dataset = TranslationDataset(train_data)
val_dataset   = TranslationDataset(val_data)
test_dataset  = TranslationDataset(test_data)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train batches : {len(train_loader)}")
print(f"Val batches   : {len(val_loader)}")

# Inspect one batch
src_batch, tgt_batch = next(iter(train_loader))
print(f"\nSource batch shape : {src_batch.shape}")
print(f"Target batch shape : {tgt_batch.shape}")

---
## Section 4: The Transformer Model

Here we build the heart of the assignment — the Transformer model.

Recall the architecture from class:

```
                     OUTPUT PROBABILITIES
                           ↑
                    [Linear + Softmax]
                           ↑
               ┌───────────────────────┐
               │   DECODER (N layers)  │
               │  - Masked Self-Attn   │
               │  - Cross-Attention    │
               │  - FFN                │
               └───────────────────────┘
                           ↑
               ┌───────────────────────┐
               │   ENCODER (N layers)  │
               │  - Self-Attention     │
               │  - FFN                │
               └───────────────────────┘
                           ↑
               [Token Embedding + Positional Encoding]
                           ↑
                      INPUT TOKENS
```

We use **`nn.Transformer`** from PyTorch, which implements the full encoder-decoder stack. Your job is to wrap it with embeddings, positional encodings, and a final output projection.

In [ ]:
# GIVEN — Positional Encoding
# This adds position information to token embeddings.
# The formula is: PE(pos, 2i) = sin(pos / 10000^(2i/d_model))
#                 PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.1, max_len=512):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)

        pe     = torch.zeros(max_len, d_model)  # shape: (max_len, d_model)
        pos    = torch.arange(0, max_len).unsqueeze(1).float()  # (max_len, 1)
        div    = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(pos * div)  # even indices
        pe[:, 1::2] = torch.cos(pos * div)  # odd indices
        pe = pe.unsqueeze(0)  # shape: (1, max_len, d_model) — batch dim

        self.register_buffer('pe', pe)  # not a parameter; saved with model

    def forward(self, x):
        # x shape: (batch, seq_len, d_model)
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)

In [ ]:
# TODO — Transformer Seq2Seq Model
# Complete the class below. The skeleton is provided; your job is to fill in
# the __init__ and forward methods.
#
# Architecture components you need:
#   - nn.Embedding (for source tokens)
#   - nn.Embedding (for target tokens) — or a shared embedding if you're feeling bold
#   - PositionalEncoding (already defined above)
#   - nn.Transformer (the main encoder-decoder stack)
#   - nn.Linear (output projection: d_model → vocab_size)
#
# Key PyTorch Transformer arguments:
#   d_model       — embedding dimension (must be divisible by nhead)
#   nhead         — number of attention heads
#   num_encoder_layers — number of encoder layers
#   num_decoder_layers — number of decoder layers
#   dim_feedforward — size of the FFN hidden layer
#   dropout       — dropout probability
#   batch_first   — set to True so input shape is (batch, seq, d_model)

class Seq2SeqTransformer(nn.Module):
    def __init__(
        self,
        vocab_size,
        d_model=256,
        nhead=8,
        num_encoder_layers=3,
        num_decoder_layers=3,
        dim_feedforward=512,
        dropout=0.1,
        max_seq_len=MAX_SEQ_LEN,
    ):
        super().__init__()

        # TODO: Store hyperparameters as attributes (e.g., self.d_model = d_model)

        # TODO: Define self.src_embedding  — nn.Embedding(vocab_size, d_model)
        # TODO: Define self.tgt_embedding  — nn.Embedding(vocab_size, d_model)
        # TODO: Define self.pos_encoding   — PositionalEncoding(d_model, dropout, max_seq_len)
        # TODO: Define self.transformer    — nn.Transformer(...) with batch_first=True
        # TODO: Define self.output_proj    — nn.Linear(d_model, vocab_size)

        pass  # Remove when implemented

    def forward(self, src, tgt, src_key_padding_mask=None, tgt_key_padding_mask=None, tgt_mask=None):
        """
        Args:
            src : (batch, src_seq_len)  — source token IDs
            tgt : (batch, tgt_seq_len)  — target token IDs (teacher-forced during training)
            src_key_padding_mask : (batch, src_seq_len) — True where tokens are PAD
            tgt_key_padding_mask : (batch, tgt_seq_len) — True where tokens are PAD
            tgt_mask             : (tgt_seq_len, tgt_seq_len) — causal mask
        Returns:
            logits : (batch, tgt_seq_len, vocab_size)
        """
        # Step 1: Embed source and scale by sqrt(d_model)
        # TODO: src_emb = self.src_embedding(src) * math.sqrt(self.d_model)

        # Step 2: Add positional encoding
        # TODO: src_emb = self.pos_encoding(src_emb)

        # Step 3: Same for target
        # TODO: tgt_emb = ...

        # Step 4: Pass through transformer
        # TODO: out = self.transformer(src_emb, tgt_emb, tgt_mask=tgt_mask,
        #                              src_key_padding_mask=src_key_padding_mask,
        #                              tgt_key_padding_mask=tgt_key_padding_mask,
        #                              memory_key_padding_mask=src_key_padding_mask)

        # Step 5: Project to vocabulary
        # TODO: logits = self.output_proj(out)

        # TODO: return logits
        pass

    def generate_causal_mask(self, sz):
        """Upper-triangular causal mask to prevent the decoder from attending to future tokens."""
        mask = torch.triu(torch.ones(sz, sz), diagonal=1).bool()
        return mask.to(DEVICE)

In [ ]:
# GIVEN — Model configuration (default experiment)
# We define a config dictionary so it's easy to swap hyperparameters

CONFIG = {
    "vocab_size"         : tokenizer.get_vocab_size(),
    "d_model"            : 256,
    "nhead"              : 8,
    "num_encoder_layers" : 3,
    "num_decoder_layers" : 3,
    "dim_feedforward"    : 512,
    "dropout"            : 0.1,
    "max_seq_len"        : MAX_SEQ_LEN,
}

model = Seq2SeqTransformer(**CONFIG).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model parameters: {total_params:,}")
print(model)

### 📝 Reflection — Section 4

1. How many parameters does your model have? What portion of them come from the embedding layers vs. the Transformer layers?
2. Why do we scale the embeddings by `sqrt(d_model)` before adding positional encodings?
3. What does `batch_first=True` do, and why does it matter?

> *(Write your answers here)*

---
## Section 5: Training

We train the model using **teacher forcing**: during training, the decoder receives the ground-truth previous token at each step (rather than its own previous prediction). This speeds up training but creates a gap with inference (where we use the model's own output).

**Loss function:** Cross-entropy over vocabulary, ignoring PAD tokens.

**Optimizer:** Adam with a **learning rate warm-up schedule** (same as the original "Attention is All You Need" paper).

In [ ]:
# GIVEN — Loss function (ignores PAD tokens)
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)

In [ ]:
# TODO — Define the optimizer
# Use Adam with lr=1e-4 and weight_decay=1e-5
# Hint: optim.Adam(model.parameters(), lr=..., weight_decay=...)

optimizer = None  # TODO: Replace with your optimizer

In [ ]:
# GIVEN — Training step helper

def make_masks(src, tgt):
    """
    Creates padding masks and causal mask.
    Padding mask: True where the token is PAD (tells attention to ignore it)
    Causal mask : prevents decoder from seeing future tokens
    """
    src_pad_mask = (src == PAD_IDX).to(DEVICE)  # (batch, src_len)
    tgt_pad_mask = (tgt == PAD_IDX).to(DEVICE)  # (batch, tgt_len)

    tgt_len      = tgt.size(1)
    causal_mask  = torch.triu(torch.ones(tgt_len, tgt_len), diagonal=1).bool().to(DEVICE)

    return src_pad_mask, tgt_pad_mask, causal_mask


def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0

    for src, tgt in tqdm(loader, desc="Training", leave=False):
        src, tgt = src.to(DEVICE), tgt.to(DEVICE)

        # Teacher forcing: decoder input is tgt[:-1], target is tgt[1:]
        tgt_input  = tgt[:, :-1]  # everything except the last token
        tgt_target = tgt[:, 1:]   # everything except the first token (SOS)

        src_pad, tgt_pad, causal = make_masks(src, tgt_input)

        optimizer.zero_grad()
        logits = model(src, tgt_input,
                       src_key_padding_mask=src_pad,
                       tgt_key_padding_mask=tgt_pad,
                       tgt_mask=causal)  # (batch, tgt_len, vocab)

        # Reshape for cross-entropy: (batch * tgt_len, vocab) vs (batch * tgt_len)
        loss = criterion(logits.reshape(-1, logits.size(-1)), tgt_target.reshape(-1))
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # Gradient clipping
        optimizer.step()
        total_loss += loss.item()

    return total_loss / len(loader)


def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for src, tgt in loader:
            src, tgt = src.to(DEVICE), tgt.to(DEVICE)
            tgt_input  = tgt[:, :-1]
            tgt_target = tgt[:, 1:]
            src_pad, tgt_pad, causal = make_masks(src, tgt_input)
            logits = model(src, tgt_input,
                           src_key_padding_mask=src_pad,
                           tgt_key_padding_mask=tgt_pad,
                           tgt_mask=causal)
            loss   = criterion(logits.reshape(-1, logits.size(-1)), tgt_target.reshape(-1))
            total_loss += loss.item()
    return total_loss / len(loader)

In [ ]:
# TODO — Training Loop
# Complete the training loop below.
# Train for N_EPOCHS epochs, saving train and val loss per epoch.
# Save the best model (lowest val loss) to 'best_model.pt'

N_EPOCHS = 10  # Start with 10; use more if you have GPU time

train_losses = []
val_losses   = []
best_val_loss = float('inf')

for epoch in range(1, N_EPOCHS + 1):
    start = time.time()

    # TODO: Call train_epoch, get train_loss
    train_loss = None  # TODO

    # TODO: Call evaluate, get val_loss
    val_loss   = None  # TODO

    # TODO: Append losses to train_losses and val_losses

    # TODO: If val_loss < best_val_loss, save model with torch.save(model.state_dict(), 'best_model.pt')

    elapsed = time.time() - start
    print(f"Epoch {epoch:02d}/{N_EPOCHS} | "
          f"Train Loss: {train_loss:.4f} | "
          f"Val Loss: {val_loss:.4f} | "
          f"Time: {elapsed:.1f}s")

In [ ]:
# TODO — Plot Training Curves
# Plot train and validation loss vs. epoch on the same graph.
# Add a legend, axis labels, and a title.
# Mark the epoch with the best validation loss with a vertical dashed line.

# TODO: Your plotting code here

### 📝 Reflection — Section 5

1. Does the training loss and validation loss follow the expected pattern? At what epoch does the model appear to converge?
2. What is **gradient clipping** (`clip_grad_norm_`) and why is it important for Transformers?
3. What is **teacher forcing**? What potential problem might it cause at inference time?

> *(Write your answers here)*

---
## Section 6: Inference and Translation

During inference, we use **greedy decoding**: at each step, pick the token with the highest probability. More advanced strategies (beam search, sampling) exist but greedy is a good starting point.

We start the decoder with `[SOS]` and stop when it generates `[EOS]` or we hit `MAX_SEQ_LEN`.

In [ ]:
# GIVEN — Greedy Decoding

def translate(model, sentence, max_len=MAX_SEQ_LEN):
    """
    Translates an English sentence to Urdu using greedy decoding.
    """
    model.eval()

    # Encode source
    src_ids = encode_sentence(sentence, max_len)
    src     = torch.tensor(src_ids).unsqueeze(0).to(DEVICE)  # (1, seq_len)

    with torch.no_grad():
        # Encode source through encoder only
        src_emb    = model.pos_encoding(model.src_embedding(src) * math.sqrt(model.d_model))
        src_pad    = (src == PAD_IDX)
        memory     = model.transformer.encoder(src_emb, src_key_padding_mask=src_pad)

        # Decode autoregressively
        tgt_tokens = [SOS_IDX]
        for _ in range(max_len):
            tgt      = torch.tensor(tgt_tokens).unsqueeze(0).to(DEVICE)
            tgt_emb  = model.pos_encoding(model.tgt_embedding(tgt) * math.sqrt(model.d_model))
            tgt_len  = tgt.size(1)
            causal   = torch.triu(torch.ones(tgt_len, tgt_len), diagonal=1).bool().to(DEVICE)

            out      = model.transformer.decoder(tgt_emb, memory, tgt_mask=causal,
                                                  memory_key_padding_mask=src_pad)
            logits   = model.output_proj(out[:, -1, :])  # take last position
            next_tok = logits.argmax(dim=-1).item()
            tgt_tokens.append(next_tok)

            if next_tok == EOS_IDX:
                break

    # Decode token IDs back to Urdu text
    output_ids = [t for t in tgt_tokens if t not in (SOS_IDX, EOS_IDX, PAD_IDX)]
    translation = tokenizer.decode(output_ids)
    return translation

In [ ]:
# TODO — Load the best saved model and run translations

# Load best model
model.load_state_dict(torch.load('best_model.pt', map_location=DEVICE))

# TODO: Translate 10 English sentences from the test set
# For each one, print:
#   Source    : (English)
#   Reference : (Ground truth Urdu)
#   Prediction: (Your model's Urdu output)

# TODO: Your translation loop here

---
## Section 7: Evaluation — BLEU Score

**BLEU (Bilingual Evaluation Understudy)** is the standard automatic metric for machine translation. It measures n-gram overlap between the model's output and one or more reference translations.

- BLEU ranges from 0 to 100
- Higher is better (human translations often score 50–70)
- A score of 10–20 on a low-resource pair like EN-UR is reasonable

In [ ]:
# TODO — Compute BLEU score on the test set
# Use the sacrebleu library.
#
# Steps:
#   1. Collect all model predictions and reference translations for the test set
#      (use a subset of ~500 sentences to keep it manageable)
#   2. Use sacrebleu.corpus_bleu(hypotheses, [references]) to compute the score
#   3. Print the BLEU score
#
# Note: hypotheses is a list of strings, references is a list of lists of strings

hypotheses = []  # TODO: your model's translations
references  = []  # TODO: ground-truth Urdu translations

N_EVAL = 500  # Evaluate on this many test examples

# TODO: Loop over test_data[:N_EVAL] and populate hypotheses and references

# TODO: bleu = sacrebleu.corpus_bleu(hypotheses, [references])
# TODO: print(f"BLEU Score: {bleu.score:.2f}")

### 📝 Reflection — Section 7

1. What BLEU score did you get? Is it what you expected?
2. Look at some of the translations where the model did well and some where it did poorly. What patterns do you notice (sentence length, vocabulary, sentence type)?
3. What are the limitations of BLEU as a metric? What would be a better way to evaluate translation quality?

> *(Write your answers here)*

---
## Section 8: Hyperparameter Experiments 🔬

This is the most important section. The Transformer has many architectural hyperparameters that significantly affect performance, speed, and generalization.

**Your task:** Run at least 3 different configurations and compare them.

You must vary **at least one** of the following:
| Hyperparameter | Suggested values |
|---|---|
| `nhead` (attention heads) | 2, 4, 8, 16 |
| `num_encoder_layers` | 1, 2, 3, 6 |
| `d_model` | 128, 256, 512 |
| `dim_feedforward` | 256, 512, 1024 |
| `dropout` | 0.0, 0.1, 0.3 |

> ⚠️ **Constraint:** `d_model` must be divisible by `nhead`.

For each configuration, record: number of parameters, final training loss, final validation loss, BLEU score.

In [ ]:
# GIVEN — Experiment runner helper

def run_experiment(config, n_epochs=5, experiment_name="exp"):
    """
    Trains a model with the given config for n_epochs and returns
    a results dict with losses and parameter count.
    """
    print(f"\n{'='*60}")
    print(f"Experiment: {experiment_name}")
    print(f"Config: {config}")
    print('='*60)

    m = Seq2SeqTransformer(**config).to(DEVICE)
    opt = optim.Adam(m.parameters(), lr=1e-4, weight_decay=1e-5)

    t_losses, v_losses = [], []

    for epoch in range(1, n_epochs + 1):
        tl = train_epoch(m, train_loader, opt, criterion)
        vl = evaluate(m, val_loader, criterion)
        t_losses.append(tl)
        v_losses.append(vl)
        print(f"  Epoch {epoch}: train={tl:.4f}  val={vl:.4f}")

    n_params = sum(p.numel() for p in m.parameters() if p.requires_grad)

    return {
        "name"        : experiment_name,
        "config"      : config,
        "n_params"    : n_params,
        "train_losses": t_losses,
        "val_losses"  : v_losses,
        "best_val"    : min(v_losses),
        "model"       : m,
    }

In [ ]:
# TODO — Define your 3 experimental configurations
# Each must differ in at least one meaningful hyperparameter.
# Think carefully about WHY you chose these configurations before running.

BASE_CONFIG = {
    "vocab_size"         : tokenizer.get_vocab_size(),
    "d_model"            : 256,
    "nhead"              : 8,
    "num_encoder_layers" : 3,
    "num_decoder_layers" : 3,
    "dim_feedforward"    : 512,
    "dropout"            : 0.1,
    "max_seq_len"        : MAX_SEQ_LEN,
}

# TODO: Define EXP_CONFIG_2 (change one or more hyperparameters)
EXP_CONFIG_2 = {**BASE_CONFIG}  # Copy base and modify
# EXP_CONFIG_2["nhead"] = ???    # Example

# TODO: Define EXP_CONFIG_3
EXP_CONFIG_3 = {**BASE_CONFIG}

# Run all 3
results = []
results.append(run_experiment(BASE_CONFIG,   n_epochs=5, experiment_name="Baseline"))
results.append(run_experiment(EXP_CONFIG_2,  n_epochs=5, experiment_name="Experiment 2"))
results.append(run_experiment(EXP_CONFIG_3,  n_epochs=5, experiment_name="Experiment 3"))

In [ ]:
# TODO — Comparison Plot
# Plot the validation loss curves of all 3 experiments on one graph.
# Each line should be labeled with the experiment name.
# Include axis labels, a legend, and a title.

# TODO: Your plotting code here

In [ ]:
# TODO — Summary Table
# Print a summary table comparing all experiments:
# | Experiment | nhead | d_model | Layers | # Params | Best Val Loss |
# Use a formatted print or pandas DataFrame.

# TODO: Your summary table here

### 📝 Reflection — Section 8

1. Which configuration performed best? What do you think explains this?
2. What happened when you changed the number of attention heads? Did more heads always help?
3. Is there a trade-off between model size and validation performance? What does this suggest about overfitting?
4. If you had unlimited compute, which hyperparameter would you explore next and why?

> *(Write your answers here)*

---
## Section 9: Attention Visualization (Bonus) ⭐

One of the most powerful things about Transformers is that we can **visualize what the model is attending to**. This gives us a window into the model's internal reasoning.

In this bonus section, extract and visualize the cross-attention weights between an English source sentence and its Urdu translation.

In [ ]:
# BONUS TODO — Attention Weight Extraction and Visualization
#
# Steps:
#   1. Register a forward hook on one of the decoder cross-attention layers
#      to capture attention weights during a forward pass.
#      Use: model.transformer.decoder.layers[0].multihead_attn
#
#   2. Run a translation through the model.
#
#   3. Use plt.imshow() or seaborn.heatmap() to visualize the attention
#      matrix (rows = decoder positions / Urdu tokens, cols = encoder positions / English tokens).
#
#   4. Label the axes with the actual tokens.
#
# Hint: nn.MultiheadAttention returns (output, attn_weights) when need_weights=True.
# You may need to patch the forward call or use a hook.

attention_weights = {}

def attention_hook(module, input, output):
    # output[1] contains attention weights if need_weights=True
    if isinstance(output, tuple) and output[1] is not None:
        attention_weights['cross_attn'] = output[1].detach().cpu()

# TODO: Register hook on cross-attention layer
# TODO: Run translation
# TODO: Plot heatmap

---
## ✅ Final Submission Checklist

Before submitting, make sure:

- [ ] All `# TODO` cells are implemented and run without errors
- [ ] All Reflection sections are answered
- [ ] Training curves are plotted and labeled
- [ ] At least 3 hyperparameter experiments are run and compared
- [ ] BLEU score is reported
- [ ] The notebook runs top-to-bottom without errors (`Kernel > Restart & Run All`)

---

## 📊 Grading Rubric

| Section | Points |
|---|---|
| Section 1 — Dataset Analysis | 10 |
| Section 2 — Tokenizer Exploration | 5 |
| Section 4 — Model Implementation | 25 |
| Section 5 — Training Loop + Curves | 20 |
| Section 6 — Inference & Translations | 10 |
| Section 7 — BLEU Evaluation | 10 |
| Section 8 — Hyperparameter Experiments | 15 |
| Reflection Questions (all) | 5 |
| **Bonus** — Attention Visualization | +5 |
| **Total** | **100 (+5)** |

---

> *Good luck! You are building a real neural machine translation system — the same kind of technology that powers Google Translate and DeepL.*